# Exploratory Data Analysis: UCI Online Retail

Every number in this notebook is computed from the data when the notebook runs.
Business rules R1-R10 (see README, BUSINESS_RULES) define how rows are treated.
Amounts are in **currency units** because the dataset states no currency.

## 1. Load, validate and clean

In [ ]:
import os
import sys
from pathlib import Path

# Notebooks run from their own folder; move to the project root so imports work.
ROOT = Path.cwd()
while not (ROOT / "config").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import pandas as pd

from src.ingestion import load_file
from src.validation import validate
from src.cleaning import clean, build_product_names

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

RAW_FILE = ROOT / "data" / "raw" / "Online Retail.xlsx"
raw, ingestion = load_file(RAW_FILE)
print(ingestion)

### Data-quality summary (validation, on the raw data)

In [ ]:
validation = validate(raw)
print(f"Rows: {validation.total_rows:,} | candidate sale rows: {validation.sale_rows:,} | blocking: {validation.has_blocking_errors}")
validation.to_frame()

### Cleaning and audit trail

In [ ]:
df, cleaning_report = clean(raw)
print(f"Rows in: {cleaning_report.rows_in:,} -> rows out: {cleaning_report.rows_out:,}")
cleaning_report.to_frame()

In [ ]:
product_names = build_product_names(df)
print(f"Cleaned rows: {len(df):,} | sale rows: {int(df['is_sale'].sum()):,} | cancelled rows: {int(df['is_cancelled'].sum()):,}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Rows without CustomerID: {int((~df['has_customer']).sum()):,} ({(~df['has_customer']).mean():.1%})")
df.head()

## 2. Sales distributions
Only sale rows (rule R1) are used.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
sales = df[df["is_sale"]].copy()
print(f"Sale rows: {len(sales):,} | invoices: {sales['InvoiceNo'].nunique():,}")
sales[["Quantity", "UnitPrice", "LineTotal"]].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(sales["LineTotal"], bins=60, log_scale=True, ax=axes[0])
axes[0].set_title("Line total per sale row (log scale)")
sns.histplot(sales["Quantity"], bins=60, log_scale=True, ax=axes[1])
axes[1].set_title("Quantity per sale row (log scale)")
plt.tight_layout()
plt.show()

ordered = sales["LineTotal"].sort_values(ascending=False)
top_n = max(1, int(len(ordered) * 0.01))
share = ordered.head(top_n).sum() / ordered.sum()
print(f"The top 1% of sale lines ({top_n:,} rows) account for {share:.1%} of gross sales.")
print(f"Median line total: {sales['LineTotal'].median():.2f} | mean: {sales['LineTotal'].mean():.2f}")

## 3. Products
Product revenue excludes shipping and fees (rule R5); those are shown separately.

In [ ]:
def name_of(code_):
    return product_names.get(code_, "(no description)")

by_product = sales.groupby("StockCode").agg(revenue=("LineTotal", "sum"), units=("Quantity", "sum"),
                                            invoices=("InvoiceNo", "nunique"))
by_product["name"] = [name_of(c) for c in by_product.index]
print(f"Distinct products sold: {len(by_product):,}")
print(f"Gross product sales: {by_product['revenue'].sum():,.2f} currency units")
print("Top 10 by revenue")
display(by_product.sort_values("revenue", ascending=False).head(10)[["name", "revenue", "units", "invoices"]].round(2))
print("Top 10 by units")
display(by_product.sort_values("units", ascending=False).head(10)[["name", "units", "revenue", "invoices"]].round(2))

In [ ]:
top = by_product.sort_values("revenue", ascending=False).head(10).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh([n[:35] for n in top["name"]], top["revenue"])
plt.title("Top 10 products by revenue")
plt.xlabel("Currency units")
plt.tight_layout()
plt.show()

cum = by_product["revenue"].sort_values(ascending=False).cumsum() / by_product["revenue"].sum()
n80 = int((cum < 0.8).sum()) + 1
print(f"{n80:,} products ({n80 / len(by_product):.1%} of all products sold) make up 80% of product revenue.")

In [ ]:
fees = df[~df["is_product"] & ~df["is_cancelled"] & (df["UnitPrice"] > 0) & (df["Quantity"] > 0)]
fee_table = fees.groupby("StockCode").agg(rows=("LineTotal", "size"), amount=("LineTotal", "sum"))
fee_table["name"] = [name_of(c) for c in fee_table.index]
print(f"Shipping, fees and vouchers (not product revenue): {fee_table['amount'].sum():,.2f} currency units")
fee_table.sort_values("amount", ascending=False).round(2)

## 4. Countries
Revenue is gross sales from sale rows (rule R1). Country labels are kept as supplied.

In [ ]:
by_country = sales.groupby("Country").agg(revenue=("LineTotal", "sum"), invoices=("InvoiceNo", "nunique"),
                                          units=("Quantity", "sum"))
by_country["share_pct"] = by_country["revenue"] / by_country["revenue"].sum() * 100
by_country = by_country.sort_values("revenue", ascending=False)
print(f"Countries with sales: {len(by_country)}")
top_country = by_country.index[0]
print(f"Largest market: {top_country} with {by_country['share_pct'].iloc[0]:.1f}% of gross sales")
display(by_country.head(10).round(2))

In [ ]:
non_top = by_country.drop(index=top_country).head(10).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh(non_top.index, non_top["revenue"])
plt.title(f"Top 10 markets excluding {top_country}")
plt.xlabel("Currency units")
plt.tight_layout()
plt.show()

## 5. Time patterns
The data ends on 2011-12-09, so December 2011 is a partial month.

In [ ]:
monthly = sales.groupby("InvoiceMonth").agg(revenue=("LineTotal", "sum"), invoices=("InvoiceNo", "nunique"))
returns_m = df[df["is_cancelled"]].groupby("InvoiceMonth")["LineTotal"].sum().rename("returns")
monthly = monthly.join(returns_m).fillna(0)
monthly["net_revenue"] = monthly["revenue"] + monthly["returns"]
last_day = df["InvoiceDate"].max()
print(f"Last invoice date: {last_day}; the final month ({monthly.index[-1]}) is partial.")
display(monthly.round(2))
monthly[["revenue", "net_revenue"]].plot(kind="line", marker="o", figsize=(10, 4), title="Monthly gross and net revenue")
plt.ylabel("Currency units")
plt.tight_layout()
plt.show()

In [ ]:
s = sales.assign(weekday=sales["InvoiceDate"].dt.day_name(), hour=sales["InvoiceDate"].dt.hour)
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = s.groupby("weekday")["LineTotal"].sum().reindex(order)
print("Weekdays with no sales:", [d for d in order if pd.isna(by_day[d]) or by_day[d] == 0])
by_hour = s.groupby("hour")["LineTotal"].sum()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_day.fillna(0).plot(kind="bar", ax=axes[0], title="Revenue by weekday")
by_hour.plot(kind="bar", ax=axes[1], title="Revenue by hour of day")
plt.tight_layout()
plt.show()
print(f"Peak hour: {int(by_hour.idxmax())}:00 | peak weekday: {by_day.idxmax()}")

## 6. Customers
Only sale rows with a CustomerID (rule R6). Rows without one are excluded here and counted.

In [ ]:
cust_sales = sales[sales["has_customer"]]
missing_share = 1 - len(cust_sales) / len(sales)
print(f"Sale rows without CustomerID (excluded here): {len(sales) - len(cust_sales):,} ({missing_share:.1%} of sale rows)")
by_cust = cust_sales.groupby("CustomerID").agg(orders=("InvoiceNo", "nunique"), spend=("LineTotal", "sum"))
print(f"Customers: {len(by_cust):,}")
print(f"Median orders per customer: {by_cust['orders'].median():.0f} | mean: {by_cust['orders'].mean():.2f}")
print(f"Median spend: {by_cust['spend'].median():.2f} | mean: {by_cust['spend'].mean():.2f}")
repeat = (by_cust["orders"] > 1).mean()
print(f"Customers with more than one order: {repeat:.1%}")
ordered = by_cust["spend"].sort_values(ascending=False)
top_n = max(1, int(len(ordered) * 0.10))
print(f"The top 10% of customers ({top_n:,}) account for {ordered.head(top_n).sum() / ordered.sum():.1%} of customer spend.")
display(by_cust.sort_values("spend", ascending=False).head(10).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(by_cust["orders"], bins=40, log_scale=True, ax=axes[0])
axes[0].set_title("Orders per customer (log scale)")
sns.histplot(by_cust["spend"], bins=40, log_scale=True, ax=axes[1])
axes[1].set_title("Spend per customer (log scale)")
plt.tight_layout()
plt.show()

## 7. Computed observations
Every sentence below is generated from the data when the notebook runs.

In [ ]:
inv_totals = sales.groupby("InvoiceNo")["LineTotal"].sum()
returns_value = df.loc[df["is_cancelled"], "LineTotal"].sum()
gross = sales["LineTotal"].sum()
obs = [
    f"Gross sales are {gross:,.2f} currency units across {len(inv_totals):,} invoices; average order value is {inv_totals.mean():,.2f}.",
    f"Returns (cancelled invoices) total {returns_value:,.2f}, or {abs(returns_value) / gross:.1%} of gross sales; net revenue is {gross + returns_value:,.2f}.",
    f"{top_country} accounts for {by_country['share_pct'].iloc[0]:.1f}% of gross sales across {len(by_country)} countries.",
    f"{n80:,} of {len(by_product):,} products ({n80 / len(by_product):.1%}) make up 80% of product revenue.",
    f"{missing_share:.1%} of sale rows have no CustomerID, so customer figures cover only the rest.",
    f"{repeat:.1%} of identified customers placed more than one order.",
]
for line in obs:
    print("-", line)